# Spatial Sensitivity Analysis via Sliding Window Masking

For each coordinate (x1, y1, x2, y2), we measure which image regions affect the prediction most.

**Method:** Prefix-forced masking with KL divergence.
1. Generate baseline bbox with full image
2. For each masked image variant, force the same prefix tokens up to the target coordinate
3. Measure KL divergence between baseline and masked logit distributions
4. Aggregate into spatial heatmaps

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import json, re
from pathlib import Path
from PIL import Image, ImageFilter
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

DATA_DIR = Path('/efs/user_folders/dnshalam/datasets/lvis')
model_name = 'Qwen/Qwen3-VL-8B-Instruct'
processor = AutoProcessor.from_pretrained(model_name)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map='auto',
)
model.eval()

with open(DATA_DIR / 'lvis_validation.json') as f:
    data = json.load(f)
print(f'Loaded model and {len(data)} samples')

In [ ]:
# ── Configuration ──
PATCH_SIZE = 14  # Qwen3-VL ViT patch size
WINDOW_SIZE = 56  # mask window in pixels (4x4 patches)
STRIDE = 28       # stride in pixels (2 patches)
MASK_MODE = 'blur'  # 'blur' or 'mean'
BLUR_RADIUS = 20
TOP_K = 100  # top-K tokens for approximate KL divergence

In [ ]:
def mask_image(image, x, y, window_size, mode='blur', blur_radius=20):
    """Apply mask to image region. Returns masked PIL image."""
    img = image.copy()
    w, h = img.size
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(w, x + window_size), min(h, y + window_size)
    
    if mode == 'blur':
        blurred = image.filter(ImageFilter.GaussianBlur(radius=blur_radius))
        region = blurred.crop((x1, y1, x2, y2))
    elif mode == 'mean':
        arr = np.array(image)
        mean_val = arr.mean(axis=(0, 1)).astype(np.uint8)
        region = Image.new('RGB', (x2 - x1, y2 - y1), tuple(mean_val))
    else:
        region = Image.new('RGB', (x2 - x1, y2 - y1), (0, 0, 0))
    
    img.paste(region, (x1, y1))
    return img


def find_coord_token_ranges(records):
    """Find token index ranges for each coordinate."""
    tokens = [r['token_str'] for r in records]
    text = ''.join(tokens)
    m = re.search(r'"bbox_2d"\s*:\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]', text)
    if not m:
        return None
    ranges = {}
    pos = 0
    for i, t in enumerate(tokens):
        ts, te = pos, pos + len(t)
        for j, name in enumerate(['x1', 'y1', 'x2', 'y2']):
            cs, ce = m.start(j+1), m.end(j+1)
            if ts < ce and te > cs:
                if name not in ranges:
                    ranges[name] = []
                ranges[name].append(i)
        pos = te
    return ranges

In [ ]:
def get_baseline(model, processor, image, prompt_text):
    """Step 1: Generate baseline and capture token sequence + logits."""
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt_text},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    n_input = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=128,
                                return_dict_in_generate=True, output_scores=True)
    
    generated_ids = output.sequences[0, n_input:]
    gen_text = processor.decode(generated_ids, skip_special_tokens=False)
    
    # Build records with baseline logit distributions
    records = []
    for i, score in enumerate(output.scores):
        logits = score[0].float()
        probs = F.softmax(logits, dim=-1)
        tok_id = generated_ids[i].item()
        records.append({
            'token_id': tok_id,
            'token_str': processor.tokenizer.decode(tok_id),
            'logits': logits.cpu(),
            'probs': probs.cpu(),
        })
    
    return records, generated_ids, n_input, gen_text


def get_masked_logits(model, processor, masked_image, prompt_text, 
                      prefix_ids, n_input, target_step):
    """Step 2: Prefix-forced forward pass with masked image.
    
    Forces the model to use baseline tokens as prefix up to target_step,
    then extracts the logit distribution at target_step.
    """
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': masked_image},
        {'type': 'text', 'text': prompt_text},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[masked_image], return_tensors='pt').to(model.device)
    
    # Construct input: prompt + forced prefix tokens up to target_step
    forced_prefix = prefix_ids[:target_step].unsqueeze(0).to(model.device)
    input_ids = torch.cat([inputs['input_ids'], forced_prefix], dim=-1)
    attn_mask = torch.ones_like(input_ids)
    
    static_inputs = {k: v for k, v in inputs.items() 
                     if k not in ('input_ids', 'attention_mask')}
    
    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attn_mask, **static_inputs)
    
    logits = out.logits[0, -1, :].float().cpu()
    probs = F.softmax(logits, dim=-1)
    return logits, probs

In [ ]:
def kl_divergence_topk(p_base, p_masked, top_k=100):
    """Approximate KL(p_base || p_masked) using top-K tokens from baseline."""
    topk_vals, topk_ids = torch.topk(p_base, k=top_k)
    p_b = topk_vals
    p_m = p_masked[topk_ids].clamp(min=1e-10)
    return (p_b * torch.log(p_b / p_m)).sum().item()


def compute_sensitivity_map(model, processor, image, prompt_text,
                            records, generated_ids, n_input, coord_ranges,
                            window_size=WINDOW_SIZE, stride=STRIDE,
                            mask_mode=MASK_MODE, top_k=TOP_K):
    """Compute spatial sensitivity maps for each coordinate."""
    w, h = image.size
    grid_x = list(range(0, w - window_size + 1, stride))
    grid_y = list(range(0, h - window_size + 1, stride))
    
    sensitivity = {}
    for name in ['x1', 'y1', 'x2', 'y2']:
        if name in coord_ranges:
            sensitivity[name] = np.zeros((len(grid_y), len(grid_x)))
    
    total = len(grid_x) * len(grid_y)
    print(f'Computing {total} masked variants ({len(grid_x)}x{len(grid_y)} grid)...')
    
    for iy, y in enumerate(tqdm(grid_y, desc='Rows')):
        for ix, x in enumerate(grid_x):
            masked_img = mask_image(image, x, y, window_size, mode=mask_mode)
            
            for name, steps in coord_ranges.items():
                target_step = steps[0]  # first digit of coordinate
                base_probs = records[target_step]['probs']
                
                _, masked_probs = get_masked_logits(
                    model, processor, masked_img, prompt_text,
                    generated_ids, n_input, target_step)
                
                kl = kl_divergence_topk(base_probs, masked_probs, top_k=top_k)
                sensitivity[name][iy, ix] = kl
    
    return sensitivity, grid_x, grid_y

In [ ]:
# ── Run on a sample ──
SAMPLE_IDX = 0  # <-- change to explore

item = data[SAMPLE_IDX]
prompt_text = item['conversations'][0]['value'].replace('<image>\n', '')
image = Image.open(item['image']).convert('RGB')

# Parse GT
gt_match = re.search(r'"bbox_2d"\s*:\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]',
                     item['conversations'][1]['value'])
gt_coords = [int(gt_match.group(i)) for i in range(1, 5)] if gt_match else None

print(f'Prompt: {prompt_text}')
print(f'Image size: {image.size}')
print(f'GT: {gt_coords}')

# Step 1: Baseline
records, generated_ids, n_input, gen_text = get_baseline(model, processor, image, prompt_text)
coord_ranges = find_coord_token_ranges(records)

pred_match = re.search(r'"bbox_2d"\s*:\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]', gen_text)
pred_coords = [int(pred_match.group(i)) for i in range(1, 5)] if pred_match else None

print(f'Pred: {pred_coords}')
print(f'Coord ranges: {coord_ranges}')

In [ ]:
# Step 2-3: Compute sensitivity maps
sensitivity, grid_x, grid_y = compute_sensitivity_map(
    model, processor, image, prompt_text,
    records, generated_ids, n_input, coord_ranges)

In [ ]:
# Step 5: Visualize heatmaps
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
w, h = image.size

for col, name in enumerate(['x1', 'y1', 'x2', 'y2']):
    ax = axes[col]
    ax.imshow(image)
    
    if name in sensitivity:
        hmap = sensitivity[name]
        # Normalize to [0, 1]
        if hmap.max() > 0:
            hmap = hmap / hmap.max()
        # Upscale to image resolution via bicubic
        hmap_resized = np.array(Image.fromarray(hmap.astype(np.float32)).resize(
            (w, h), Image.BICUBIC))
        ax.imshow(hmap_resized, alpha=0.6, cmap='jet')
    
    # GT box (green dashed)
    if gt_coords:
        gx1, gy1, gx2, gy2 = [c/1000 for c in gt_coords]
        ax.add_patch(patches.Rectangle((gx1*w, gy1*h), (gx2-gx1)*w, (gy2-gy1)*h,
            lw=2, ec='lime', fc='none', ls='--'))
    # Pred box (red solid)
    if pred_coords:
        px1, py1, px2, py2 = [c/1000 for c in pred_coords]
        ax.add_patch(patches.Rectangle((px1*w, py1*h), (px2-px1)*w, (py2-py1)*h,
            lw=2, ec='red', fc='none'))
    
    ax.set_title(f'{name} sensitivity', fontsize=13, fontweight='bold')
    ax.axis('off')

fig.suptitle(f'Spatial Sensitivity (KL Divergence) - {prompt_text}\n'
             f'Green=GT, Red=Pred | Window={WINDOW_SIZE}px, Stride={STRIDE}px, Mode={MASK_MODE}',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-digit sensitivity (optional, slower) ──
# Uncomment to see sensitivity for each digit of each coordinate

# coord_name = 'x1'  # change as needed
# if coord_name in coord_ranges and len(coord_ranges[coord_name]) > 1:
#     steps = coord_ranges[coord_name]
#     fig, axes = plt.subplots(1, len(steps), figsize=(5*len(steps), 5))
#     if len(steps) == 1: axes = [axes]
#     
#     for digit_idx, step in enumerate(steps):
#         digit_sens = np.zeros((len(grid_y), len(grid_x)))
#         base_probs = records[step]['probs']
#         
#         for iy, y in enumerate(tqdm(grid_y, desc=f'{coord_name} digit {digit_idx}')):
#             for ix, x in enumerate(grid_x):
#                 masked_img = mask_image(image, x, y, WINDOW_SIZE, mode=MASK_MODE)
#                 _, masked_probs = get_masked_logits(
#                     model, processor, masked_img, prompt_text,
#                     generated_ids, n_input, step)
#                 digit_sens[iy, ix] = kl_divergence_topk(base_probs, masked_probs)
#         
#         ax = axes[digit_idx]
#         ax.imshow(image)
#         if digit_sens.max() > 0:
#             digit_sens /= digit_sens.max()
#         hmap = np.array(Image.fromarray(digit_sens.astype(np.float32)).resize((w,h), Image.BICUBIC))
#         ax.imshow(hmap, alpha=0.6, cmap='jet')
#         ax.set_title(f'{coord_name} digit {digit_idx}: "{records[step]["token_str"]}"')
#         ax.axis('off')
#     plt.tight_layout()
#     plt.show()